# TP04 — Testing and Code Quality for an ML Project

**ESILV — Machine Learning Operations**  
**Duration:** 90 minutes  
**Dataset/context:** IBM Telco Customer Churn  
**Starting point:** your own TP02/TP03 repository  
**Format:** independent implementation with short hints and instructor support

> A colleague changes your preprocessing code. Training still runs, but how do you know that prediction behavior has not been broken?

This notebook keeps the subject of the TP. After each section, a ✅ block says where it was done in this repository.

| Deliverable | File |
|---|---|
| Summary | `reports/tp04_summary.md` |
| Answers | `evidence/tp04/answers.md` |

In [ ]:
# Commands below run from the project root (two levels above this notebook).
import os
from pathlib import Path

if not Path("pyproject.toml").exists():
    os.chdir(Path.cwd().parents[1])
print(Path.cwd())

## Your mission

Protect your existing ML project with a small, meaningful test suite and repeatable local quality checks. Then install pre-commit hooks so that lightweight checks run before a commit is accepted.

Reuse your preprocessing and training code. You are not asked to improve model scores or implement a new algorithm. The objective is evidence that important behavior is preserved.

This is a student brief, not a reference implementation. Replace placeholders, adapt imports and implement the fixtures and functions that your project needs.

### Learning outcomes

- Translate an input/output contract into useful assertions.
- Use pytest fixtures, expected exceptions and temporary files.
- Distinguish behavioral tests, static checks and formatting.
- Interpret coverage without treating it as proof of correctness.
- Install and demonstrate pre-commit hooks.
- Explain how these local checks will become CI checks in TP05.

## Scope and timing

| Time | Task |
|---|---|
| 0–10 min | Set up tools and define the behaviors to protect |
| 10–45 min | Implement four focused tests using small synthetic data |
| 45–55 min | Deliberately introduce a regression, observe failure and repair it |
| 55–65 min | Run Ruff and inspect coverage |
| 65–80 min | Configure and demonstrate pre-commit |
| 80–90 min | Collect evidence and discuss Quality Gates |

**Core:** pytest, Ruff, pre-commit and a short coverage observation.  
**Instructor demonstration or extension:** SonarQube Community Build. No VM, cloud service or paid account is required for the core.

## 1. Prepare the project and define its contract — 10 minutes

From the root of your existing project:

```bash
uv add --dev pytest pytest-cov ruff pre-commit
uv run pytest --version
uv run ruff --version
uv run pre-commit --version
```

Keep the resolved versions in your lockfile. If these dependencies already exist, preserve a consistent configuration rather than creating another project.

Add test files to your existing structure. Suggested locations:

| Path | Responsibility |
|---|---|
| `tests/conftest.py` | Shared small synthetic data and fitted-pipeline fixtures |
| `tests/test_input_contract.py` | Missing columns and permitted input variation |
| `tests/test_prediction.py` | Probability/output behavior |
| `tests/test_persistence.py` | Save/reload agreement |
| `pyproject.toml` | Tool settings, alongside existing package metadata |
| `.pre-commit-config.yaml` | Commit-hook configuration |
| `reports/tp04_summary.md` | Evidence and explanations |

Adapt package imports to your Cookiecutter or src layout. Your package must be importable through the project environment; do not hardcode an absolute path or modify `sys.path` in tests to hide packaging problems.

Write a short input/output contract before testing:

- The six TP02 features are required; neither `Churn` nor `customerID` is a model input.
- Numeric inputs follow your documented parsing rules.
- Permitted missing values and unknown categories follow the preprocessing policy you chose.
- A nonempty batch receives one prediction/probability row per customer.
- Loading a saved fitted pipeline preserves its predictions.

Keep `Churn = Yes` as encoded class `1`. Document the class order when testing probabilities.

**Checkpoint:** name four realistic defects your checks should detect. “The program runs” is not a sufficient contract.

> ✅ **In this repo:** Contract in `reports/tp04_summary.md` (section 1). Tools in `pyproject.toml` (dev group) and `uv.lock`.

In [ ]:
!uv run pytest --version
!uv run ruff --version
!uv run pre-commit --version

## 2. Build four focused tests — 35 minutes

### Make a small fixture first

Create a synthetic DataFrame with the six agreed input columns and a separate binary target. Use approximately 12–20 invented customers, with both classes represented. Include enough variation for your existing preprocessing recipe and estimator.

Use a modest classifier and fixed random state where applicable. Fit your actual pipeline builder on this fixture. Tests should run in seconds, without reading the full Telco CSV, downloading data, accessing a network or starting MLflow.

If importing your training module starts training or contacts the tracking server, refactor it. Keep execution behind a command entry point or a `if __name__ == "__main__":` guard. Test reusable functions without invoking the whole experiment workflow.

**Fixture hint:**

```python
import pytest

@pytest.fixture
def sample_features():
    # TODO: return a small DataFrame with the six agreed columns.
    ...

@pytest.fixture
def fitted_pipeline(sample_features):
    # TODO: construct your existing pipeline, create a target with both classes,
    # and fit using synthetic training data only.
    ...
```

A pytest fixture supplies reusable test inputs/setup. It is different from a production dataset or saved model. Use a copy when mutating a fixture's DataFrame.

### Test A — a required feature is missing

Remove one required feature from a valid inference batch. Assert that your validation/prediction boundary rejects it with the documented exception and an informative message.

**Hint:**

```python
def test_missing_required_feature_is_rejected(fitted_pipeline, sample_features):
    broken = sample_features.drop(columns=["Contract"])

    # TODO: call your prediction/validation entry point inside pytest.raises.
    # Assert the exception type and a stable, meaningful message fragment.
    ...
```

Research `pytest.raises(ExpectedException, match="...")`. Catch the expected exception, not a broad `Exception` that could hide unrelated failures. A failed assertion must not be replaced by printing a warning.

### Test B — supported input variation

Make a copy of valid inputs and introduce a numeric missing value and a categorical value absent from the fixture's training data.

For TP02's usual policy—imputation and `OneHotEncoder(handle_unknown="ignore")`—assert that prediction succeeds, returns the expected number of rows and produces valid outputs. Verify that the test category really was absent from training.

If your documented policy intentionally rejects unknown categories, test that explicit rejection instead and explain the choice. Do not change the test merely to match an accidental crash.

**Question:** does successful prediction on an unknown category prove the prediction is accurate? Why not?

### Test C — probability contract

For your classifier supporting `predict_proba`, check:

- Output shape agrees with batch size and number of classes.
- Values are finite and between 0 and 1.
- Each row sums to approximately 1.
- The positive-class score is selected using `classes_`.

**Partial assertion hint:**

```python
probabilities = fitted_pipeline.predict_proba(sample_features)

assert probabilities.shape == (...)  # TODO: expected batch size and class count.
assert np.isfinite(probabilities).all()
# TODO: check bounds, row sums with tolerance, and positive-class lookup.
```

Use NumPy's approximate comparison helpers where appropriate. Probability validity does not establish calibration or predictive quality. Avoid asserting an exact ROC-AUC or accuracy on this tiny fixture.

### Test D — serialization preserves behavior

Use pytest's `tmp_path` to save your fitted preprocessing-and-classifier pipeline with joblib. Reload it, then compare predictions or probabilities on the same inputs.

**Hint:**

```python
def test_reload_preserves_predictions(
    tmp_path, fitted_pipeline, sample_features
):
    artifact = tmp_path / "pipeline.joblib"
    before = ...  # TODO: obtain predictions or probabilities.
    # TODO: save, reload and predict on the same inputs without refitting.
    after = ...
    # TODO: compare class labels exactly, or probabilities within a tolerance.
```

Only load your own trusted artifact. The temporary file avoids overwriting your TP02/TP03 model. This check tests serialization behavior in the current test process; TP03's fresh-process demonstration covers an additional environment/import boundary.

Run your suite:

```bash
uv run pytest -q
```

**Checkpoint:** at least four meaningful tests pass. For each, explain the real defect it could catch. Checking only that an object exists or that a mocked function was called is insufficient.

> ✅ **In this repo:** Fixtures: `tests/conftest.py`. Tests A/B: `tests/test_input_contract.py`, C: `tests/test_prediction.py`, D: `tests/test_persistence.py`. Inference boundary: `src/churnlab/predict.py`.

In [ ]:
!uv run pytest -q

## 3. Prove a test detects a regression — 10 minutes

Temporarily introduce one realistic defect in your own project, for example:

- Change unknown-category handling from the documented permissive policy to rejection.
- Remove a required-column validation check where doing so changes the contract.
- Save only the classifier instead of the complete pipeline in your persistence helper.

Choose a change that your existing tests should detect. If the persistence test serializes directly with joblib, alter the relevant pipeline behavior rather than pretending it tests a separate save helper.

Run the relevant test, read the traceback and record the failed assertion. Restore/fix the implementation and rerun the suite.

Do not demonstrate failure merely by inserting `assert False`, changing the expected answer arbitrarily or deleting a test.

**Checkpoint:** one failure caused by a behavioral regression, followed by a passing result after repair. Keep your final submitted code working.

> ✅ **In this repo:** `handle_unknown="error"` in `src/churnlab/pipeline.py` → test B failed, then restored. Logs: `evidence/tp04/regression_diff.txt`, `regression_failure.txt`, `regression_fixed.txt`.

In [ ]:
!cat evidence/tp04/regression_failure.txt

## 4. Static checks and coverage — 10 minutes

### Ruff: inspect, repair, verify

Run against your package and tests. Replace paths to match your project:

```bash
uv run ruff check src/telco_churn tests
uv run ruff format --check src/telco_churn tests
```

For a root-package layout, use `telco_churn tests`. Include your relevant scripts if applicable.

Start with a small rule set, for example `E4`, `E7`, `E9`, `F` and `I`, configured under `[tool.ruff.lint]` in the existing `pyproject.toml`. Research what the groups mean; preserve existing compatible settings. The formatter is separate from lint rules.

Review available automatic fixes before accepting them:

```bash
uv run ruff check --fix src/telco_churn tests
uv run ruff format src/telco_churn tests
git diff
```

Rerun tests after editing. Do not blanket-ignore findings or exclude your entire source package to obtain green output.

### Coverage: find a gap

```bash
uv run pytest --cov=telco_churn --cov-report=term-missing --cov-report=xml:coverage.xml
```

Replace `telco_churn` with your importable package name. Inspect one uncovered, relevant behavior and propose a useful test for it. You do not have to reach an arbitrary percentage during this TP.

**Questions:** what does an executed line tell you? Can a test execute a line without asserting the right behavior? Which third-party code is outside the intended coverage scope?

> ✅ **In this repo:** Ruff config in `pyproject.toml`. Before/after: `evidence/tp04/ruff_check_before.txt`, `ruff_check_after.txt`. Coverage: `evidence/tp04/coverage.txt`.

In [ ]:
!uv run ruff check src scripts tests
!uv run ruff format --check src scripts tests
!uv run pytest -q --cov --cov-report=term-missing

## 5. Run lightweight checks before commit — 15 minutes

Create `.pre-commit-config.yaml`. Enable Ruff lint and format hooks, then optionally add trailing-whitespace and end-of-file checks.

**Configuration hint — incomplete:**

```yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: REPLACE_WITH_A_REAL_PINNED_TAG
    hooks:
      - id: ruff-check
        args: [--fix]
        types_or: [python, pyi]
      - id: ruff-format
        types_or: [python, pyi]
```

Replace the placeholder with a released tag compatible with your installed Ruff; use the same Ruff version for command-line and hooks. See the official integration documentation. Put lint fixes before formatting. If using the separate `pre-commit-hooks` repository for whitespace checks, pin a real release there too.

For this first configuration, scope Python hooks to your project package, scripts and tests if other unrelated generated files are present. Do not scan virtual environments or MLflow storage.

Install and run:

```bash
uv run pre-commit validate-config
uv run pre-commit install
uv run pre-commit run --all-files
```

The first run may download hook environments and takes longer; internet access is needed. The configuration belongs in Git, but hook installation is local: a peer must run the install command after cloning.

### Demonstrate a blocked commit

Introduce a formatting issue in a tracked Python file without changing its intended behavior. Stage that file and attempt a commit.

Observe the hook result. When hooks modify files, inspect `git diff`, stage the reviewed changes again and retry the commit. Do not use `--no-verify` to complete the exercise.

**Checkpoint:** show a blocked/modified commit attempt and a successful retry.

Keep full training and MLflow server startup out of hooks. Run pytest manually in this TP; TP05 will run it in CI. Local hooks can be bypassed, so they cannot replace server-side checks.

> ✅ **In this repo:** `.pre-commit-config.yaml`. Blocked commit and retry: `evidence/tp04/precommit_blocked.txt`, `precommit_retry.txt`.

In [ ]:
!uv run pre-commit run --all-files

## 6. Handover and Quality Gate discussion — 10 minutes

Update the README with dependency setup, test, lint, formatting, coverage and hook-installation commands. Ask a peer to explain which command detects which kind of issue.

Keep `.venv/`, `.pytest_cache/`, `.ruff_cache/`, `.coverage`, `coverage.xml` and optional `htmlcov/` out of Git. Submit requested report evidence separately through the course platform.

### Instructor demonstration / optional extension — SonarQube Community Build

**Use the free, open-source SonarQube Community Build, hosted locally or on an instructor-provided instance. No paid subscription or commercial trial is needed for this exercise.** Do not sign up for a paid product to complete this TP.

Use the instructor's prepared instance or screenshots when available. Installing a server is outside the 90-minute core. Local container installation is an optional extension and uses your computer's resources.

| Capability | Community Build in this course |
|---|---|
| Main-branch code analysis and issue browsing | Included |
| Import externally generated coverage | Included |
| Quality Gate pass/fail result | Included |
| Fail a CI job when the Quality Gate fails | Supported, with CI integration and server connectivity |
| Official branch and pull-request-specific analysis | Commercial feature; not required |

**Required versus optional:** complete the local pytest, Ruff and pre-commit tasks for core assessment. Participate in the Quality Gate discussion; running your own SonarQube server and submitting its screenshots are optional.

Observe:

1. One concrete issue and its location.
2. The difference between a code issue and test coverage.
3. The Quality Gate result and the actual condition responsible.
4. The difference between analysis completing and the Quality Gate passing.

SonarQube imports externally generated coverage; it does not run your pytest suite to produce coverage. For Python, investigate `sonar.python.coverage.reportPaths=coverage.xml`. Source/test paths must match your actual layout.

A Quality Gate is a policy evaluating analysis results. Its thresholds and definition of “new code” affect the outcome; do not assume every project uses the same rules. A CI job can wait for the gate and fail when it fails, for example through a Quality Gate check or `sonar.qualitygate.wait=true`. Analysis success and gate success are separate outcomes. Making a CI check block a merge is a further repository-policy step.

Do not confuse this CI capability with official pull-request-specific analysis, which is outside Community Build. For TP05, Ruff, pytest and the security checks form the required PR checks; SonarQube integration remains an extension.

**Local connectivity:** a GitHub-hosted runner cannot directly reach the SonarQube server at `localhost` on your laptop. Do not expose your local server just for this exercise. Use the local scanner for the optional TP04 analysis; any later CI connection requires an instructor-approved reachable instance or a suitably configured runner.

**Discussion:** if tests pass, what can still be wrong? If a Quality Gate passes, does that prove the model is useful, fair or secure? Explain the limits.

> ✅ **In this repo:** README section "TP04", Quality Gate reflection in `reports/tp04_summary.md` (section 7). SonarQube not run (optional).

## Deliverables

Submit your repository/source archive plus `reports/tp04_summary.md` with:

| Evidence | Expected content |
|---|---|
| Tests | At least four meaningful behavior checks and a brief test-to-risk mapping |
| Regression demonstration | One real failure, the reason, the fix and passing rerun |
| Ruff | Passing lint/format checks on the agreed project scope |
| Coverage | Actual result, one uncovered behavior and a proposed useful test |
| pre-commit | Pinned configuration, blocked/modified commit and successful retry |
| README | Commands another student can execute |
| Quality Gate reflection | Short distinction between tests, static analysis, coverage and policy |

Screenshots or short text logs are sufficient. Do not invent scores or findings. SonarQube screenshots are optional and are not required for full core marks.

### Assessment

| Criterion | Points |
|---|---:|
| Four meaningful tests using appropriate small fixtures and assertions | 4 |
| Genuine regression detected and fixed | 2 |
| Ruff configuration and passing checks | 1 |
| Coverage interpretation and a useful proposed check | 1 |
| Working pre-commit setup and commit demonstration | 1 |
| Clear README, evidence and limits of quality checks | 1 |
| **Total** | **10** |

A higher coverage percentage alone does not earn a better grade. Reliable checks and explanations matter.

## Troubleshooting hints

| Symptom | First check |
|---|---|
| No tests collected | File/function names start with `test_`; run from the repository root |
| Package cannot be imported | Check project packaging and environment installation |
| Tests contact MLflow or start full training | Remove import-time execution; test reusable functions |
| Fixture training fails with one class | Include both labels in the synthetic training fixture |
| Unknown-category test passes unexpectedly | Was that category truly absent during fitting? |
| pre-commit reports “files were modified” | Review, re-stage and rerun; this is expected for fixers |
| Hooks skip a new test file | Is it tracked/staged and inside the hook's file scope? |
| CLI and hooks disagree | Compare pinned Ruff versions, rule settings and scanned files |
| No coverage data | Check the importable package name and that tests execute your code |

## Optional extensions

- Parameterize multiple missing-column cases with `pytest.mark.parametrize`.
- Check that inference does not change a learned imputer/scaler statistic.
- Add a focused split test proving no customer IDs overlap.
- Add one test for whitespace-only versus malformed nonblank numeric values.
- Perform SonarQube analysis using the instructor's prepared environment.

## Official documentation

- [pytest: fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html)
- [pytest: expected exceptions](https://docs.pytest.org/en/stable/how-to/assert.html)
- [pytest: temporary files](https://docs.pytest.org/en/stable/how-to/tmp_path.html)
- [pytest-cov: reports](https://pytest-cov.readthedocs.io/en/stable/reporting.html)
- [Ruff integrations and hooks](https://docs.astral.sh/ruff/integrations/)
- [pre-commit](https://pre-commit.com/)
- [SonarQube Community Build: coverage import](https://docs.sonarsource.com/sonarqube-community-build/analyzing-source-code/test-coverage/test-coverage-parameters)
- [SonarQube Community Build: Quality Gates](https://docs.sonarsource.com/sonarqube-community-build/quality-standards-administration/managing-quality-gates/introduction-to-quality-gates)

**Next TP:** reuse these checks in GitHub Actions, then add dependency auditing, secret scanning and Dependabot. Container scanning with Trivy comes after you have a container image.